In [1]:

import os
import re
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

nltk.download('punkt')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:

def read_test(testset):
    id_gts = {}
    with open(testset, 'r', encoding='utf8') as fh:
        for line in fh:
            fields = line.strip().split('\t')
            tweetid = fields[0]
            gt = fields[1]
            id_gts[tweetid] = gt
    return id_gts

def confusion(id_preds, testset, classifier):
    id_gts = read_test(testset)

    gts = ['positive', 'negative', 'neutral']
    conf = {}
    for c1 in gts:
        conf[c1] = {}
        for c2 in gts:
            conf[c1][c2] = 0

    for tweetid, gt in id_gts.items():
        if tweetid in id_preds:
            pred = id_preds[tweetid]
        else:
            pred = 'neutral'
        conf[pred][gt] += 1

    print(f"Confusion matrix for: {testset} ({classifier})")
    print(''.ljust(12) + '  '.join(gts))
    for c1 in gts:
        print(c1.ljust(12), end='')
        total = float(sum(conf[c1].values()))
        for c2 in gts:
            if total > 0:
                print(f"{conf[c1][c2]/total:.3f}     ", end='')
            else:
                print("0.000     ", end='')
        print('')
    print('')

def evaluate(id_preds, testset, classifier):
    id_gts = read_test(testset)

    acc_by_class = {}
    for gt in ['positive', 'negative', 'neutral']:
        acc_by_class[gt] = {'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0}

    ok = 0
    for tweetid, gt in id_gts.items():
        pred = id_preds[tweetid] if tweetid in id_preds else 'neutral'
        if gt == pred:
            ok += 1
            acc_by_class[gt]['tp'] += 1
        else:
            acc_by_class[gt]['fn'] += 1
            acc_by_class[pred]['fp'] += 1

    semevalmacro = {'p': 0, 'r': 0, 'f1': 0}
    for cat in ['positive', 'negative', 'neutral']:
        tp = acc_by_class[cat]['tp']
        fp = acc_by_class[cat]['fp']
        fn = acc_by_class[cat]['fn']

        p = float(tp)/(tp + fp) if (tp + fp) > 0 else 0
        r = float(tp)/(tp + fn) if (tp + fn) > 0 else 0
        f1 = 2*p*r/(p + r) if (p + r) > 0 else 0

        if cat in ['positive', 'negative']:
            semevalmacro['p'] += p
            semevalmacro['r'] += r
            semevalmacro['f1'] += f1

    # macro-F1 over positive & negative
    semevalmacrof1 = semevalmacro['f1'] / 2
    print(f"{testset} ({classifier}): {semevalmacrof1:.3f}")

In [3]:
def load_data(file_path):
    data_rows = []
    with open(file_path, mode='r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t')
            if len(parts) == 3:
                tweet_id, label, text = parts
                data_rows.append([tweet_id, label, text])
    df = pd.DataFrame(data_rows, columns=['tweet_id','label','text'])
    return df


In [4]:

def clean_tweet(text):
    text = text.lower()
    text = re.sub(r'(https?://\S+)|(\bwww\.\S+)', '<URL>', text)
    text = re.sub(r'@\w+', '<USER>', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    tokens = word_tokenize(text)
    return tokens

def text_to_clean_str(text):
    tokens = clean_tweet(text)
    return " ".join(tokens)


In [6]:
train_path = "/kaggle/input/nlp-assignment2-dataset/twitter-training-data.txt"
dev_path   = "/kaggle/input/nlp-assignment2-dataset/twitter-dev-data.txt"
test1_path = "/kaggle/input/nlp-assignment2-dataset/twitter-test1.txt"
test2_path = "/kaggle/input/nlp-assignment2-dataset/twitter-test2.txt"
test3_path = "/kaggle/input/nlp-assignment2-dataset/twitter-test3.txt"

train_df = load_data(train_path)
dev_df   = load_data(dev_path)
test1_df = load_data(test1_path)
test2_df = load_data(test2_path)
test3_df = load_data(test3_path)


In [8]:
train_texts = [text_to_clean_str(t) for t in train_df['text']]
y_train     = train_df['label']

tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=10000)
X_train = tfidf.fit_transform(train_texts)

lr_clf = LogisticRegression(max_iter=1000, solver='liblinear')
lr_clf.fit(X_train, y_train)

LogisticRegression(max_iter=1000, solver='liblinear')

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
import numpy as np


f1_macro = make_scorer(f1_score, average='macro')  

params = {
    "C": [0.01, 0.1, 1.0, 10.0],
    "penalty": ["l2"],  
    "solver": ["liblinear"]
}

lr = LogisticRegression(max_iter=1000)
grid = GridSearchCV(
    estimator=lr,
    param_grid=params,
    scoring=f1_macro,
    cv=3,  # 3-fold cross validation
    verbose=1,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)


best_lr = grid.best_estimator_


Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best Params: {'C': 10.0, 'penalty': 'l2', 'solver': 'liblinear'}
Best CV Score: 0.6110802033719012


In [10]:
def generate_predictions_dict(model, vectorizer, test_df):
    test_texts = [text_to_clean_str(t) for t in test_df['text']]
    X_test = vectorizer.transform(test_texts)
    preds = model.predict(X_test)

    id_preds = {}
    for i, tweet_id in enumerate(test_df['tweet_id']):
        id_preds[tweet_id] = preds[i]
    return id_preds

In [13]:
test1_preds = generate_predictions_dict(best_lr, tfidf, test1_df)
evaluate(test1_preds, "/kaggle/input/nlp-assignment2-dataset/twitter-test1.txt", classifier="LogReg_Tuned")
confusion(test1_preds, "/kaggle/input/nlp-assignment2-dataset/twitter-test1.txt", classifier="LogReg_Tuned")

test1_preds = generate_predictions_dict(best_lr, tfidf, test2_df)
evaluate(test1_preds, "/kaggle/input/nlp-assignment2-dataset/twitter-test2.txt", classifier="LogReg_Tuned")
confusion(test1_preds, "/kaggle/input/nlp-assignment2-dataset/twitter-test2.txt", classifier="LogReg_Tuned")

test1_preds = generate_predictions_dict(best_lr, tfidf, test3_df)
evaluate(test1_preds, "/kaggle/input/nlp-assignment2-dataset/twitter-test3.txt", classifier="LogReg_Tuned")
confusion(test1_preds, "/kaggle/input/nlp-assignment2-dataset/twitter-test3.txt", classifier="LogReg_Tuned")


/kaggle/input/nlp-assignment2-dataset/twitter-test1.txt (LogReg_Tuned): 0.565
Confusion matrix for: /kaggle/input/nlp-assignment2-dataset/twitter-test1.txt (LogReg_Tuned)
            positive  negative  neutral
positive    0.722     0.060     0.218     
negative    0.165     0.628     0.207     
neutral     0.245     0.149     0.605     

/kaggle/input/nlp-assignment2-dataset/twitter-test2.txt (LogReg_Tuned): 0.589
Confusion matrix for: /kaggle/input/nlp-assignment2-dataset/twitter-test2.txt (LogReg_Tuned)
            positive  negative  neutral
positive    0.771     0.051     0.178     
negative    0.183     0.626     0.191     
neutral     0.336     0.099     0.565     

/kaggle/input/nlp-assignment2-dataset/twitter-test3.txt (LogReg_Tuned): 0.535
Confusion matrix for: /kaggle/input/nlp-assignment2-dataset/twitter-test3.txt (LogReg_Tuned)
            positive  negative  neutral
positive    0.712     0.055     0.232     
negative    0.191     0.538     0.271     
neutral     0.316    

In [14]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC  # or SVC
from sklearn.feature_extraction.text import TfidfVectorizer


def text_to_clean_str(text):

    tokens = clean_tweet(text)
    return " ".join(tokens)

train_texts = [text_to_clean_str(t) for t in train_df['text']]
y_train     = train_df['label']

tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=10000)
X_train = tfidf.fit_transform(train_texts)


def generate_predictions_dict(model, vectorizer, df):
    test_texts = [text_to_clean_str(t) for t in df['text']]
    X_test = vectorizer.transform(test_texts)
    preds = model.predict(X_test)

    id_preds = {}
    for i, tweet_id in enumerate(df['tweet_id']):
        id_preds[tweet_id] = preds[i]
    return id_preds

############################
# Train Naïve Bayes
############################
nb_clf = MultinomialNB()
nb_clf.fit(X_train, y_train)

############################
# Train SVM
############################
svm_clf = LinearSVC()
svm_clf.fit(X_train, y_train)

############################
# Evaluate on test sets
############################

for (name, model) in [("NaiveBayes", nb_clf), ("SVM", svm_clf)]:
    for test_file, df_test in [
        ("/kaggle/input/nlp-assignment2-dataset/twitter-test1.txt", test1_df),
        ("/kaggle/input/nlp-assignment2-dataset/twitter-test2.txt", test2_df),
        ("/kaggle/input/nlp-assignment2-dataset/twitter-test3.txt", test3_df)
    ]:
        preds_dict = generate_predictions_dict(model, tfidf, df_test)
        evaluate(preds_dict, test_file, classifier=name)
        confusion(preds_dict, test_file, classifier=name)


/kaggle/input/nlp-assignment2-dataset/twitter-test1.txt (NaiveBayes): 0.416
Confusion matrix for: /kaggle/input/nlp-assignment2-dataset/twitter-test1.txt (NaiveBayes)
            positive  negative  neutral
positive    0.688     0.069     0.244     
negative    0.135     0.697     0.169     
neutral     0.267     0.188     0.544     

/kaggle/input/nlp-assignment2-dataset/twitter-test2.txt (NaiveBayes): 0.463
Confusion matrix for: /kaggle/input/nlp-assignment2-dataset/twitter-test2.txt (NaiveBayes)
            positive  negative  neutral
positive    0.749     0.058     0.194     
negative    0.036     0.857     0.107     
neutral     0.330     0.137     0.533     

/kaggle/input/nlp-assignment2-dataset/twitter-test3.txt (NaiveBayes): 0.410
Confusion matrix for: /kaggle/input/nlp-assignment2-dataset/twitter-test3.txt (NaiveBayes)
            positive  negative  neutral
positive    0.690     0.098     0.211     
negative    0.167     0.694     0.139     
neutral     0.318     0.154     0

In [15]:
import collections

def build_vocab(all_texts, max_vocab_size=5000):
    freq = collections.Counter()
    for tokens in all_texts:
        freq.update(tokens)

    # Sort by frequency
    most_common = freq.most_common(max_vocab_size - 2) 
    idx2word = ['<PAD>', '<UNK>'] + [w for w, _ in most_common]
    word2idx = {w: i for i, w in enumerate(idx2word)}

    return word2idx, idx2word


In [16]:
train_tokens = [clean_tweet(txt) for txt in train_df['text']]
dev_tokens   = [clean_tweet(txt) for txt in dev_df['text']]

all_tokens = train_tokens + dev_tokens
word2idx, idx2word = build_vocab(all_tokens, max_vocab_size=5000)

vocab_size = len(idx2word) 
print("Vocab size:", vocab_size)


Vocab size: 5000


In [17]:
import numpy as np

def load_glove_embeddings(glove_path, word2idx, embedding_dim=100):
    vocab_size = len(word2idx)
    embedding_matrix = np.random.uniform(-0.05, 0.05, (vocab_size, embedding_dim))

    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            vals = line.strip().split()
            word = vals[0]
            vector = vals[1:]
            if len(vector) != embedding_dim:
                continue
            if word in word2idx:
                idx = word2idx[word]
                embedding_matrix[idx] = np.array(vector, dtype=float)
    return embedding_matrix


In [18]:
glove_path = "/kaggle/input/nlp-assignment2-dataset/glove.6B.100d.txt"  
embedding_dim = 100
embedding_matrix = load_glove_embeddings(glove_path, word2idx, embedding_dim)
print("Embedding matrix shape:", embedding_matrix.shape)


Embedding matrix shape: (5000, 100)


In [19]:
import torch
from torch.utils.data import Dataset, DataLoader

label_map = {'negative': 0, 'neutral': 1, 'positive': 2} 

def tokens_to_indices(tokens, word2idx, max_len=30):
    idxs = []
    for w in tokens:
        if w in word2idx:
            idxs.append(word2idx[w])
        else:
            idxs.append(word2idx['<UNK>'])
    # pad or truncate
    idxs = idxs[:max_len]
    idxs += [word2idx['<PAD>']] * (max_len - len(idxs))
    return idxs

class TwitterDataset(Dataset):
    def __init__(self, df, word2idx, max_len=30):
        self.samples = []
        for _, row in df.iterrows():
            text_tokens = clean_tweet(row['text'])
            x_idxs = tokens_to_indices(text_tokens, word2idx, max_len)
            label_str = row['label']
            y_label = label_map[label_str]
            self.samples.append((x_idxs, y_label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x_idxs, y_label = self.samples[idx]
        return torch.LongTensor(x_idxs), torch.LongTensor([y_label])


In [20]:
train_dataset = TwitterDataset(train_df, word2idx, max_len=30)
dev_dataset   = TwitterDataset(dev_df, word2idx, max_len=30)

train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)
dev_loader    = DataLoader(dev_dataset,   batch_size=32, shuffle=False)


In [21]:
import torch.nn as nn

class LSTMSentiment(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, embedding_matrix):
        super(LSTMSentiment, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        self.embedding.weight = nn.Parameter(torch.tensor(embedding_matrix, dtype=torch.float32))
        
        self.embedding.weight.requires_grad = False
        
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc   = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)            
        lstm_out, (h_n, c_n) = self.lstm(embedded)
        final_feat = h_n[-1]  
        logits = self.fc(final_feat)  
        return logits


In [22]:
hidden_dim = 128
num_classes = 3

model = LSTMSentiment(
    vocab_size = vocab_size,
    embed_dim = embedding_dim,
    hidden_dim = hidden_dim,
    num_classes = num_classes,
    embedding_matrix = embedding_matrix
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


LSTMSentiment(
  (embedding): Embedding(5000, 100)
  (lstm): LSTM(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=3, bias=True)
)

In [23]:
import torch.optim as optim
import numpy as np

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_epoch(model, train_loader):
    model.train()
    total_loss = 0
    correct, total = 0, 0
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)          
        y_batch = y_batch.squeeze(1).to(device)  
        
        optimizer.zero_grad()
        logits = model(x_batch)  
        
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    avg_loss = total_loss / len(train_loader)
    acc = correct / total
    return avg_loss, acc

def eval_epoch(model, dev_loader):
    model.eval()
    total_loss = 0
    correct, total = 0, 0
    
    with torch.no_grad():
        for x_batch, y_batch in dev_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.squeeze(1).to(device)
            
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    
    avg_loss = total_loss / len(dev_loader)
    acc = correct / total
    return avg_loss, acc

num_epochs = 6
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader)
    dev_loss, dev_acc = eval_epoch(model, dev_loader)
    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.3f} | "
          f"Dev Loss: {dev_loss:.4f}, Dev Acc: {dev_acc:.3f}")


Epoch 1/6 | Train Loss: 0.8903, Train Acc: 0.568 | Dev Loss: 0.8033, Dev Acc: 0.632
Epoch 2/6 | Train Loss: 0.7946, Train Acc: 0.632 | Dev Loss: 0.7755, Dev Acc: 0.635
Epoch 3/6 | Train Loss: 0.7647, Train Acc: 0.648 | Dev Loss: 0.7471, Dev Acc: 0.659
Epoch 4/6 | Train Loss: 0.7377, Train Acc: 0.661 | Dev Loss: 0.7341, Dev Acc: 0.678
Epoch 5/6 | Train Loss: 0.7131, Train Acc: 0.675 | Dev Loss: 0.7209, Dev Acc: 0.671
Epoch 6/6 | Train Loss: 0.6859, Train Acc: 0.687 | Dev Loss: 0.7279, Dev Acc: 0.676


In [24]:
def predict_dataset(model, df, word2idx, max_len, device):
    model.eval()
    
    dataset = TwitterDataset(df, word2idx, max_len)
    
    preds_list = []
    
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    with torch.no_grad():
        for x_batch, _ in loader:  
            x_batch = x_batch.to(device)
            logits = model(x_batch)  
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            preds_list.extend(preds)
    
    return preds_list


In [25]:
inv_label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}


In [26]:
def build_prediction_dict(df, preds_int):
    id_preds = {}
    for i, tweet_id in enumerate(df['tweet_id']):
        label_str = inv_label_map[preds_int[i]]
        id_preds[tweet_id] = label_str
    return id_preds


In [27]:
for i, (test_file, df_test) in enumerate([
    ("/kaggle/input/nlp-assignment2-dataset/twitter-test1.txt", test1_df),
    ("/kaggle/input/nlp-assignment2-dataset/twitter-test2.txt", test2_df),
    ("/kaggle/input/nlp-assignment2-dataset/twitter-test3.txt", test3_df)
]):
    
    preds_int = predict_dataset(model, df_test, word2idx, max_len=30, device=device)
    
    id_preds = build_prediction_dict(df_test, preds_int)
    
    evaluate(id_preds, test_file, classifier="LSTM_GloVe")
    confusion(id_preds, test_file, classifier="LSTM_GloVe")


/kaggle/input/nlp-assignment2-dataset/twitter-test1.txt (LSTM_GloVe): 0.540
Confusion matrix for: /kaggle/input/nlp-assignment2-dataset/twitter-test1.txt (LSTM_GloVe)
            positive  negative  neutral
positive    0.753     0.035     0.212     
negative    0.122     0.782     0.096     
neutral     0.248     0.173     0.579     

/kaggle/input/nlp-assignment2-dataset/twitter-test2.txt (LSTM_GloVe): 0.545
Confusion matrix for: /kaggle/input/nlp-assignment2-dataset/twitter-test2.txt (LSTM_GloVe)
            positive  negative  neutral
positive    0.783     0.038     0.179     
negative    0.156     0.781     0.062     
neutral     0.340     0.125     0.535     

/kaggle/input/nlp-assignment2-dataset/twitter-test3.txt (LSTM_GloVe): 0.534
Confusion matrix for: /kaggle/input/nlp-assignment2-dataset/twitter-test3.txt (LSTM_GloVe)
            positive  negative  neutral
positive    0.780     0.053     0.167     
negative    0.154     0.675     0.172     
neutral     0.300     0.141     0

In [28]:
torch.save(model.state_dict(), "lstm_model.pt")

In [29]:
model = LSTMSentiment(vocab_size, embedding_dim, hidden_dim, num_classes, embedding_matrix)
model.load_state_dict(torch.load("lstm_model.pt"))
model.to(device)
model.eval()


<ipython-input-29-dc5a7ece54d6>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("lstm_model.pt"))


LSTMSentiment(
  (embedding): Embedding(5000, 100)
  (lstm): LSTM(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=3, bias=True)
)

In [30]:
class BiLSTMSentiment(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 embedding_matrix=None, freeze_emb=True, 
                 num_layers=1, dropout=0.3):
        super(BiLSTMSentiment, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        if embedding_matrix is not None:
            self.embedding.weight = nn.Parameter(
                torch.tensor(embedding_matrix, dtype=torch.float32)
            )
        self.embedding.weight.requires_grad = not freeze_emb

        # LSTM (bidirectional)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=(dropout if num_layers > 1 else 0.0), 
            bidirectional=True
        )
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)

        lstm_out, (h_n, c_n) = self.lstm(embedded)
        
        forward_hidden = h_n[-2, :, :]   
        backward_hidden = h_n[-1, :, :]  

        final_feat = torch.cat((forward_hidden, backward_hidden), dim=1)  

        final_feat = self.dropout(final_feat)
        logits = self.fc(final_feat)  
        return logits


In [31]:
import torch.optim as optim
import copy

# Hyperparams
vocab_size    = len(word2idx)
embed_dim     = 100      
hidden_dim    = 128
num_classes   = 3        
num_layers    = 2
dropout       = 0.3
freeze_emb    = True     

# Instantiate the BiLSTM
model = BiLSTMSentiment(
    vocab_size, 
    embed_dim, 
    hidden_dim, 
    num_classes,
    embedding_matrix=embedding_matrix,
    freeze_emb=freeze_emb,
    num_layers=num_layers,
    dropout=dropout
)
model.to(device)

# Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10
best_dev_loss = float('inf')
best_model_weights = copy.deepcopy(model.state_dict())
patience = 3
epochs_no_improve = 0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.squeeze(1).to(device)  

        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc = correct / total

    model.eval()
    dev_total_loss = 0.0
    dev_correct = 0
    dev_total = 0
    
    with torch.no_grad():
        for x_batch, y_batch in dev_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.squeeze(1).to(device)

            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            dev_total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            dev_correct += (preds == y_batch).sum().item()
            dev_total += y_batch.size(0)

    dev_loss = dev_total_loss / len(dev_loader)
    dev_acc = dev_correct / dev_total

    print(f"Epoch {epoch+1}/{num_epochs} |"
          f" Train Loss: {train_loss:.4f}, Acc: {train_acc:.3f} "
          f"| Dev Loss: {dev_loss:.4f}, Acc: {dev_acc:.3f}")

    # Early Stopping
    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        best_model_weights = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stopping triggered!")
            break

# Load best weights
model.load_state_dict(best_model_weights)
print("Loaded best model with dev loss:", best_dev_loss)


Epoch 1/10 | Train Loss: 0.8634, Acc: 0.588 | Dev Loss: 0.7741, Acc: 0.641
Epoch 2/10 | Train Loss: 0.7838, Acc: 0.638 | Dev Loss: 0.7556, Acc: 0.644
Epoch 3/10 | Train Loss: 0.7564, Acc: 0.654 | Dev Loss: 0.7281, Acc: 0.672
Epoch 4/10 | Train Loss: 0.7277, Acc: 0.669 | Dev Loss: 0.7209, Acc: 0.674
Epoch 5/10 | Train Loss: 0.6958, Acc: 0.685 | Dev Loss: 0.7139, Acc: 0.679
Epoch 6/10 | Train Loss: 0.6627, Acc: 0.702 | Dev Loss: 0.7098, Acc: 0.678
Epoch 7/10 | Train Loss: 0.6256, Acc: 0.724 | Dev Loss: 0.7285, Acc: 0.675
Epoch 8/10 | Train Loss: 0.5843, Acc: 0.745 | Dev Loss: 0.7527, Acc: 0.671
Epoch 9/10 | Train Loss: 0.5369, Acc: 0.766 | Dev Loss: 0.7773, Acc: 0.650
Early stopping triggered!
Loaded best model with dev loss: 0.70977668582447


In [32]:
inv_label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}

def predict_dataset(model, df, word2idx, max_len=30, device='cpu'):
    model.eval()
    dataset = TwitterDataset(df, word2idx, max_len)  # your dataset class
    loader = DataLoader(dataset, batch_size=32, shuffle=False)

    preds_list = []
    with torch.no_grad():
        for x_batch, _ in loader:
            x_batch = x_batch.to(device)
            logits = model(x_batch)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            preds_list.extend(preds)
    return preds_list

def build_prediction_dict(df, preds_int):

    id_preds = {}
    for i, tweet_id in enumerate(df['tweet_id']):
        label_str = inv_label_map[preds_int[i]]
        id_preds[tweet_id] = label_str
    return id_preds


test1_df = load_data("/kaggle/input/nlp-assignment2-dataset/twitter-test1.txt")
test2_df = load_data("/kaggle/input/nlp-assignment2-dataset/twitter-test2.txt")
test3_df = load_data("/kaggle/input/nlp-assignment2-dataset/twitter-test3.txt")

for (filename, df_test) in [
    ("/kaggle/input/nlp-assignment2-dataset/twitter-test1.txt", test1_df),
    ("/kaggle/input/nlp-assignment2-dataset/twitter-test2.txt", test2_df),
    ("/kaggle/input/nlp-assignment2-dataset/twitter-test3.txt", test3_df)
]:
    test_preds_int = predict_dataset(model, df_test, word2idx, max_len=30, device=device)
    id_preds = build_prediction_dict(df_test, test_preds_int)
    evaluate(id_preds, filename, classifier="BiLSTM") 
    confusion(id_preds, filename, classifier="BiLSTM")


/kaggle/input/nlp-assignment2-dataset/twitter-test1.txt (BiLSTM): 0.603
Confusion matrix for: /kaggle/input/nlp-assignment2-dataset/twitter-test1.txt (BiLSTM)
            positive  negative  neutral
positive    0.781     0.043     0.175     
negative    0.145     0.700     0.155     
neutral     0.239     0.143     0.618     

/kaggle/input/nlp-assignment2-dataset/twitter-test2.txt (BiLSTM): 0.612
Confusion matrix for: /kaggle/input/nlp-assignment2-dataset/twitter-test2.txt (BiLSTM)
            positive  negative  neutral
positive    0.824     0.037     0.139     
negative    0.144     0.721     0.135     
neutral     0.330     0.102     0.568     

/kaggle/input/nlp-assignment2-dataset/twitter-test3.txt (BiLSTM): 0.582
Confusion matrix for: /kaggle/input/nlp-assignment2-dataset/twitter-test3.txt (BiLSTM)
            positive  negative  neutral
positive    0.796     0.056     0.149     
negative    0.168     0.601     0.231     
neutral     0.299     0.116     0.585     



In [33]:
def get_misclassifications(id_preds, df):
    misclassified = []
    for _, row in df.iterrows():
        tweet_id = row['tweet_id']
        gold = row['label']
        pred = id_preds.get(tweet_id, 'neutral')  
        if gold != pred:
            misclassified.append((tweet_id, gold, pred, row['text']))
    return misclassified


In [34]:
preds_dict = generate_predictions_dict(lr_clf, tfidf, test1_df)

misclassified = get_misclassifications(preds_dict, test1_df)
print(f"Found {len(misclassified)} misclassified tweets out of {len(test1_df)}")

for i in range(10):
    tweet_id, gold, pred, text = misclassified[i]
    print("-"*50)
    print("Tweet ID:", tweet_id)
    print("Gold:", gold, " | Pred:", pred)
    print("Text:", text)


Found 1203 misclassified tweets out of 3531
--------------------------------------------------
Tweet ID: 768006053969268950
Gold: negative  | Pred: neutral
Text: @Dont__KAY_me omg same I was reading it in school after PSSAS and I just sat there crying
--------------------------------------------------
Tweet ID: 420747042670198316
Gold: negative  | Pred: positive
Text: Miyagi just got banned from yoga. He was caught him sniffing the sphincter of the girl in front of him. There may be police involvement!
--------------------------------------------------
Tweet ID: 865097686752334952
Gold: positive  | Pred: neutral
Text: "Hey friends, DST ends Sunday, 11/4. Just giving you a heads-up to set your clocks back 1hr before bed Saturday evening. :)"
--------------------------------------------------
Tweet ID: 170392754891582126
Gold: neutral  | Pred: positive
Text: Them Seniors gone beat the Juniors tomorrow.
--------------------------------------------------
Tweet ID: 677748222480342835
Gold: 

In [38]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.squeeze(1).to(device)  # shape: (batch_size,)

        # Forward pass
        logits = model(x_batch)
        loss = criterion(logits, y_batch)

        # Backprop + update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track metrics
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total
    return avg_loss, acc


In [39]:
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.squeeze(1).to(device)

            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item()

            preds = logits.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total
    return avg_loss, acc


In [40]:
def evaluate_return_f1(id_preds, testset):
    id_gts = read_test(testset)

    acc_by_class = {}
    for gt in ['positive', 'negative', 'neutral']:
        acc_by_class[gt] = {'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0}

   
    for tweetid, gt_label in id_gts.items():
        pred_label = id_preds.get(tweetid, 'neutral')  
        if gt_label == pred_label:
            acc_by_class[gt_label]['tp'] += 1
        else:
            acc_by_class[gt_label]['fn'] += 1
            acc_by_class[pred_label]['fp'] += 1

   
    semevalmacro = {'p': 0, 'r': 0, 'f1': 0}
    for cat in ['positive', 'negative', 'neutral']:
        tp = acc_by_class[cat]['tp']
        fp = acc_by_class[cat]['fp']
        fn = acc_by_class[cat]['fn']

        p = float(tp) / (tp + fp) if (tp + fp) > 0 else 0
        r = float(tp) / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

        if cat in ['positive', 'negative']:
            semevalmacro['p'] += p
            semevalmacro['r'] += r
            semevalmacro['f1'] += f1

    
    semevalmacrof1 = semevalmacro['f1'] / 2.0
    return semevalmacrof1


In [41]:
import torch
import copy

def run_experiment(hidden_dim, dropout, learning_rate, epochs):
    
    model = BiLSTMSentiment(
        vocab_size=len(word2idx),
        embed_dim=100,       
        hidden_dim=hidden_dim,
        num_classes=3,
        embedding_matrix=embedding_matrix,
        freeze_emb=True,      
        num_layers=2,
        dropout=dropout
    )
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = torch.nn.CrossEntropyLoss()

    best_dev_f1 = 0.0
    best_model_weights = None

    # training loop
    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        dev_loss, dev_acc = eval_epoch(model, dev_loader, criterion, device)


        dev_preds_int = predict_dataset(model, dev_df, word2idx, max_len=30, device=device)
        dev_id_preds = build_prediction_dict(dev_df, dev_preds_int)  

        dev_f1 = evaluate_return_f1(dev_id_preds, "/kaggle/input/nlp-assignment2-dataset/twitter-dev-data.txt")


        if dev_f1 > best_dev_f1:
            best_dev_f1 = dev_f1
            best_model_weights = copy.deepcopy(model.state_dict())

    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)
    return model, best_dev_f1


best_overall_f1 = 0.0
best_config = None
for hidden_dim in [64, 128]:
    for dropout in [0.2, 0.3]:
        for lr in [5e-4, 1e-3]:
            print(f"Testing config: hidden_dim={hidden_dim}, dropout={dropout}, lr={lr}")
            model, dev_f1 = run_experiment(hidden_dim, dropout, lr, epochs=5)
            print(f"Config done: dev_f1={dev_f1:.3f}")
            if dev_f1 > best_overall_f1:
                best_overall_f1 = dev_f1
                best_config = (hidden_dim, dropout, lr, copy.deepcopy(model.state_dict()))

print("Best config found:", best_config, "with dev_f1=", best_overall_f1)


Testing config: hidden_dim=64, dropout=0.2, lr=0.0005
Config done: dev_f1=0.602
Testing config: hidden_dim=64, dropout=0.2, lr=0.001
Config done: dev_f1=0.626
Testing config: hidden_dim=64, dropout=0.3, lr=0.0005
Config done: dev_f1=0.603
Testing config: hidden_dim=64, dropout=0.3, lr=0.001
Config done: dev_f1=0.638
Testing config: hidden_dim=128, dropout=0.2, lr=0.0005
Config done: dev_f1=0.622
Testing config: hidden_dim=128, dropout=0.2, lr=0.001
Config done: dev_f1=0.643
Testing config: hidden_dim=128, dropout=0.3, lr=0.0005
Config done: dev_f1=0.645
Testing config: hidden_dim=128, dropout=0.3, lr=0.001
Config done: dev_f1=0.646
Best config found: (128, 0.3, 0.001, OrderedDict([('embedding.weight', tensor([[-0.0408,  0.0116,  0.0368,  ..., -0.0279, -0.0486, -0.0142],
        [ 0.0494,  0.0397, -0.0236,  ..., -0.0058, -0.0066, -0.0061],
        [-1.6269,  0.7842,  0.0260,  ...,  0.1562, -0.4688, -0.6525],
        ...,
        [ 0.2191,  0.5322, -0.1603,  ..., -0.7289,  0.0706, -0.945

In [42]:
# Re-creating a new model with the best hyperparams
best_model = BiLSTMSentiment(
    vocab_size = len(word2idx),
    embed_dim  = 100,         
    hidden_dim = 128,
    num_classes= 3,
    embedding_matrix = embedding_matrix,
    freeze_emb = True,
    num_layers = 2,
    dropout = 0.2
)

best_model.to(device)


BiLSTMSentiment(
  (embedding): Embedding(5000, 100)
  (lstm): LSTM(100, 128, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=256, out_features=3, bias=True)
)

In [43]:
optimizer = torch.optim.Adam(best_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

num_epochs = 10
for epoch in range(num_epochs):
    best_model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.squeeze(1).to(device)

        logits = best_model(x_batch)
        loss = criterion(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc = correct / total

    best_model.eval()
    dev_total_loss = 0.0
    dev_correct = 0
    dev_total = 0

    with torch.no_grad():
        for x_batch, y_batch in dev_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.squeeze(1).to(device)

            logits = best_model(x_batch)
            loss = criterion(logits, y_batch)

            dev_total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            dev_correct += (preds == y_batch).sum().item()
            dev_total += y_batch.size(0)

    dev_loss = dev_total_loss / len(dev_loader)
    dev_acc = dev_correct / dev_total

    print(f"Epoch {epoch+1}/{num_epochs} "
          f"| Train Loss: {train_loss:.4f}, Acc: {train_acc:.3f} "
          f"| Dev Loss: {dev_loss:.4f}, Acc: {dev_acc:.3f}")


Epoch 1/10 | Train Loss: 0.8617, Acc: 0.588 | Dev Loss: 0.7712, Acc: 0.634
Epoch 2/10 | Train Loss: 0.7828, Acc: 0.638 | Dev Loss: 0.7453, Acc: 0.654
Epoch 3/10 | Train Loss: 0.7508, Acc: 0.655 | Dev Loss: 0.7281, Acc: 0.663
Epoch 4/10 | Train Loss: 0.7224, Acc: 0.671 | Dev Loss: 0.7297, Acc: 0.663
Epoch 5/10 | Train Loss: 0.6886, Acc: 0.689 | Dev Loss: 0.7204, Acc: 0.673
Epoch 6/10 | Train Loss: 0.6508, Acc: 0.708 | Dev Loss: 0.7118, Acc: 0.679
Epoch 7/10 | Train Loss: 0.6097, Acc: 0.729 | Dev Loss: 0.7360, Acc: 0.667
Epoch 8/10 | Train Loss: 0.5589, Acc: 0.755 | Dev Loss: 0.7733, Acc: 0.665
Epoch 9/10 | Train Loss: 0.5031, Acc: 0.784 | Dev Loss: 0.8124, Acc: 0.665
Epoch 10/10 | Train Loss: 0.4407, Acc: 0.814 | Dev Loss: 0.9106, Acc: 0.670


In [44]:
test_files = [
    "/kaggle/input/nlp-assignment2-dataset/twitter-test1.txt",
    "/kaggle/input/nlp-assignment2-dataset/twitter-test2.txt",
    "/kaggle/input/nlp-assignment2-dataset/twitter-test3.txt"
]

for f in test_files:
    df_test = load_data(f)
    test_preds_int = predict_dataset(best_model, df_test, word2idx, max_len=30, device=device)
    id_preds = build_prediction_dict(df_test, test_preds_int)
    evaluate(id_preds, f, classifier="Best_LSTM")
    confusion(id_preds, f, classifier="Best_LSTM")


/kaggle/input/nlp-assignment2-dataset/twitter-test1.txt (Best_LSTM): 0.589
Confusion matrix for: /kaggle/input/nlp-assignment2-dataset/twitter-test1.txt (Best_LSTM)
            positive  negative  neutral
positive    0.744     0.047     0.210     
negative    0.174     0.632     0.194     
neutral     0.244     0.144     0.612     

/kaggle/input/nlp-assignment2-dataset/twitter-test2.txt (Best_LSTM): 0.625
Confusion matrix for: /kaggle/input/nlp-assignment2-dataset/twitter-test2.txt (Best_LSTM)
            positive  negative  neutral
positive    0.820     0.035     0.145     
negative    0.164     0.642     0.194     
neutral     0.328     0.096     0.575     

/kaggle/input/nlp-assignment2-dataset/twitter-test3.txt (Best_LSTM): 0.567
Confusion matrix for: /kaggle/input/nlp-assignment2-dataset/twitter-test3.txt (Best_LSTM)
            positive  negative  neutral
positive    0.763     0.060     0.177     
negative    0.198     0.525     0.277     
neutral     0.292     0.121     0.587  